In [21]:
!pip install mne -q

import os
import numpy as np
import mne
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
# Path to EEG folder in Drive
EEG_ROOT = "/content/drive/MyDrive/EEG"

# Number of Subjects Used
NUM_SUBJECTS = 50

# Only real right fist runs
RIGHT_FIST_RUNS = [3, 7, 11]

# Epoch window
TMIN = 0.0
TMAX = 4.0

# Training settings
BATCH_SIZE = 32
EPOCHS = 30
TEST_SIZE = 0.2
RANDOM_STATE = 42

In [23]:
subject_folders = sorted([
    f for f in os.listdir(EEG_ROOT)
    if os.path.isdir(os.path.join(EEG_ROOT, f)) and f.startswith("S")
])

subject_folders = subject_folders[:NUM_SUBJECTS]

print("Using subjects:")
print(subject_folders)

Using subjects:
['S001', 'S002', 'S003', 'S004', 'S005', 'S006', 'S007', 'S008', 'S009', 'S010', 'S011', 'S012', 'S013', 'S014', 'S015', 'S016', 'S017', 'S018', 'S019', 'S020', 'S021', 'S022', 'S023', 'S024', 'S025', 'S026', 'S027', 'S028', 'S029', 'S030', 'S031', 'S032', 'S033', 'S034', 'S035', 'S036', 'S037', 'S038', 'S039', 'S040', 'S041', 'S042', 'S043', 'S044', 'S045', 'S046', 'S047', 'S048', 'S049', 'S050']


In [24]:
X_all = []
y_all = []

for subject in subject_folders:
    subject_path = os.path.join(EEG_ROOT, subject)

    for run in RIGHT_FIST_RUNS:
        edf_name = f"{subject}R{run:02d}.edf"
        edf_path = os.path.join(subject_path, edf_name)

        if not os.path.exists(edf_path):
            print(f"Missing: {edf_path}")
            continue

        print(f"Loading {edf_path}")

        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)

        # EEG preprocessing
        raw.pick_types(eeg=True)
        raw.filter(1., 40., fir_design="firwin", verbose=False)

        # Extract events
        events, event_id = mne.events_from_annotations(raw, verbose=False)

        # Only rest vs right fist
        selected_event_id = {
            "T0": event_id["T0"],
            "T2": event_id["T2"]
        }

        epochs = mne.Epochs(
            raw,
            events,
            event_id=selected_event_id,
            tmin=TMIN,
            tmax=TMAX,
            baseline=None,
            preload=True,
            verbose=False
        )

        X = epochs.get_data()
        y = epochs.events[:, -1]

        # Convert labels
        y = np.where(y == event_id["T2"], 1, 0)

        X_all.append(X)
        y_all.append(y)

# Combine all data
X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

print("Raw X shape:", X_all.shape)
print("Raw y shape:", y_all.shape)

Loading /content/drive/MyDrive/EEG/S001/S001R03.edf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Loading /content/drive/MyDrive/EEG/S001/S001R07.edf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Loading /content/drive/MyDrive/EEG/S001/S001R11.edf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Loading /content/drive/MyDrive/EEG/S002/S002R03.edf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Loading /content/drive/MyDrive/EEG/S002/S002R07.edf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Loading /content/drive/MyDrive/EEG/S002/S002R11.edf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Loading /content/drive/MyDrive/EEG/S003/S003R03.edf
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Loading /content/drive/MyDrive/EEG/S003/S003R07.edf
NOTE: pick_types() is a legacy function. New 

In [25]:
# Add channel dimension for CNN
X_all = X_all[..., np.newaxis]

# Normalize per sample
mean = X_all.mean(axis=(1, 2, 3), keepdims=True)
std = X_all.std(axis=(1, 2, 3), keepdims=True)
X_all = (X_all - mean) / (std + 1e-8)

print("CNN X shape:", X_all.shape)
print("Labels:", np.unique(y_all, return_counts=True))

CNN X shape: (3364, 64, 641, 1)
Labels: (array([0, 1]), array([2250, 1114]))


In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_all
)

In [27]:
n_channels = X_train.shape[1]
n_times = X_train.shape[2]

model = models.Sequential([
    layers.Input(shape=(n_channels, n_times, 1)),

    layers.Conv2D(16, kernel_size=(1, 25), padding="same", activation="relu"),
    layers.BatchNormalization(),

    layers.Conv2D(32, kernel_size=(n_channels, 1), padding="valid", activation="relu"),
    layers.BatchNormalization(),

    layers.MaxPooling2D(pool_size=(1, 4)),
    layers.Dropout(0.25),

    layers.Conv2D(64, kernel_size=(1, 15), padding="same", activation="relu"),
    layers.BatchNormalization(),

    layers.MaxPooling2D(pool_size=(1, 4)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 64, 641, 16)    │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 64, 641, 16)    │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 1, 641, 32)     │        32,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 1, 641, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 1, 160, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 1, 160, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 1, 160, 64)     │        30,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 1, 160, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 1, 40, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 1, 40, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 2560)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │       163,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 228,417 (892.25 KB)

 Trainable params: 228,193 (891.38 KB)

 Non-trainable params: 224 (896.00 B)

In [28]:
# Calculate class weights automatically
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight = dict(zip(classes, weights))
print("Class weights:", class_weight)

# Train model with class weights
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )
    ]
)

Class weights: {np.int64(0): np.float64(0.7475), np.int64(1): np.float64(1.5101010101010102)}
Epoch 1/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 144s 2s/step - accuracy: 0.5827 - loss: 0.8316 - val_accuracy: 0.5788 - val_loss: 0.6642
Epoch 2/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.6115 - loss: 0.6458 - val_accuracy: 0.6067 - val_loss: 0.6541
Epoch 3/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.6891 - loss: 0.5716 - val_accuracy: 0.6438 - val_loss: 0.6769
Epoch 4/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 144s 2s/step - accuracy: 0.7635 - loss: 0.5120 - val_accuracy: 0.7291 - val_loss: 0.5290
Epoch 5/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 152s 2s/step - accuracy: 0.7955 - loss: 0.4390 - val_accuracy: 0.7625 - val_loss: 0.4816
Epoch 6/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.8099 - loss: 0.4043 - val_accuracy: 0.7829 - val_loss: 0.4745
Epoch 7/30
68/68 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.8439 - loss: 0.3684 - val_accuracy: 0.8200 - val_loss: 0.4202
Epoch 8/30
68/68 ━

In [29]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test accuracy:", test_acc)

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

print(confusion_matrix(y_test, y_pred))
print(classification_report(
    y_test,
    y_pred,
    target_names=["Rest", "Right fist open/close"]
))

22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 217ms/step - accuracy: 0.8098 - loss: 0.4133
Test accuracy: 0.8098068237304688


22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 345ms/step
[[376  74]
 [ 54 169]]
                       precision    recall  f1-score   support

                 Rest       0.87      0.84      0.85       450
Right fist open/close       0.70      0.76      0.73       223

             accuracy                           0.81       673
            macro avg       0.78      0.80      0.79       673
         weighted avg       0.82      0.81      0.81       673



In [30]:
model.save("/content/drive/MyDrive/right_fist_cnn_model.keras")
print("Model saved to Google Drive.")

Model saved to Google Drive.
